# Mars World Model — Wan 2.1 Renderer (Google Colab, free T4)

Runs **Wan 2.1 1.3B** on Colab's free T4 GPU to turn a `SceneSpec` (text prompt + optional Blender preview frames) into a photoreal Mars video clip.

## How to use
1. **Runtime → Change runtime type → T4 GPU** (free tier).
2. Upload your `spec.json` (produced by `mars.render.WanColabRenderer`) to `/content/`.
3. Optionally upload one or more Blender preview frames as `init_*.png` for image-to-video conditioning.
4. Run all cells.
5. Download `mars_render.mp4` at the end.

## Free-tier notes
- T4 has 15 GB VRAM — plenty for Wan 2.1 1.3B in fp16.
- A 4-second 480p clip takes roughly 3–6 minutes.
- Sessions disconnect after ~12 hours or 90 min of idle. Don't leave it overnight.

## 1. Install dependencies

In [ ]:
!pip -q install --upgrade pip
!pip -q install "diffusers>=0.32" "transformers>=4.46" "accelerate>=0.34" \
    "safetensors>=0.4" "sentencepiece" "imageio[ffmpeg]" "einops"
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Load the SceneSpec

Either upload `spec.json` from your Mac, or paste a spec inline.

In [ ]:
import json, os
from pathlib import Path

SPEC_PATH = Path("/content/spec.json")

if not SPEC_PATH.exists():
    # Inline fallback so the cell runs even without an upload.
    spec = {
        "text_prompt": (
            "Mars surface, butterscotch sky, regolith terrain, low-angle sun, "
            "thin dusty atmosphere, slow forward dolly through Jezero Crater, "
            "photorealistic, cinematic, 4K"
        ),
        "fps": 24,
        "duration_s": 4.0,
        "seed": 0,
    }
    SPEC_PATH.write_text(json.dumps(spec, indent=2))

spec = json.loads(SPEC_PATH.read_text())
print(json.dumps(spec, indent=2))

## 3. Load Wan 2.1 1.3B

Wan-AI/Wan2.1-T2V-1.3B is the lightest open Cosmos-equivalent. ~6 GB on disk, runs in fp16 on a T4.

In [ ]:
import torch
from diffusers import WanPipeline

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

pipe = WanPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.to("cuda")
pipe.enable_model_cpu_offload()  # extra safety on small VRAM
pipe.enable_vae_slicing()
print("pipeline ready")

## 4. Render

In [ ]:
import imageio.v3 as iio
import torch

fps = int(spec.get("fps", 24))
duration = float(spec.get("duration_s", 4.0))
n_frames = int(round(duration * fps))
# Wan 2.1 1.3B prefers num_frames in {49, 65, 81} at 480p — we round to nearest.
valid = [49, 65, 81]
num_frames = min(valid, key=lambda v: abs(v - n_frames))
print(f"requested ~{n_frames} frames → using {num_frames} (model-supported)")

generator = torch.Generator(device="cuda").manual_seed(int(spec.get("seed", 0)))

out = pipe(
    prompt=spec["text_prompt"],
    negative_prompt=(
        "earth, blue sky, green vegetation, water, oceans, rivers, clouds, "
        "city, buildings, people, blurry, distorted, watermark, text"
    ),
    height=480,
    width=832,
    num_frames=num_frames,
    num_inference_steps=30,
    guidance_scale=6.0,
    generator=generator,
)

frames = out.frames[0]  # list of PIL images
print(f"got {len(frames)} frames, size {frames[0].size}")

## 5. Save MP4 + download

In [ ]:
import numpy as np
import imageio.v3 as iio

OUT = "/content/mars_render.mp4"
arr = np.stack([np.asarray(f) for f in frames], axis=0)
iio.imwrite(OUT, arr, fps=fps, codec="libx264", quality=8)
print("wrote", OUT, arr.shape)

from google.colab import files
files.download(OUT)

## 6. Optional: upload to your Drive for the Mac to pick up

If you want the loop to be hands-free, mount Drive and drop the result in a known folder. The local `WanColabRenderer.render()` can then poll that folder for outputs.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil; shutil.copy(OUT, '/content/drive/MyDrive/MarsWorldModel/renders/')